# Forecast Wind Power with kNN Regressor

The file `kNN_real_time_script.py` produces a forecast of wind power generation for a user-specified window. If the forecast window falls within the already downloaded data range, the script uses saved data for training and forecasting. If the window lies beyond this range, the script (in theory) **downloads and cleans the relevant wind power output and weather data to train the model**, along with weather forecast data for the prediction window. However, due to the absence of an API key for `power_api_call`, we currently restrict forecasts to dates **on or before `2025-06-22`**.

### Key Features of `kNN_real_time_script.py`

- The script defines a class `kNN_forecast` with default parameters `pca_comp = 35` and `n_nbr = 5`, which can be modified via the constructor.
- The model is trained using 60 days of historical data by default. This can be adjusted (between 10 and 180 days) via the `days_to_train_on` argument.
- The user is allowed to choose a maximum length of forecasting window by passing the parameter `window_max_length` (default set to 30) to the constructor.
- After creating an object of this class, the forecast window can be set using one of the following methods:
  - `input()` – interactively asks the user for input via terminal or Jupyter notebook.
  - `manual_input(start_date: str, window: int)` – allows manual assignment via arguments.
  - `set_input(start_date: str, window: int)` – used as a callback interface for the Dash app (`dash_app.py`).
- Calling the `forecast(display: bool = True)` method updates:
  - The `predicted_values` attribute.
  - A `plotly.graph_objects.Figure` object.
  - By default, the graph object is automatically displayed. Toggle to `False` is not required to display.
- The plot shows:
  - The data the model was trained on.
  - The predicted values for the forecast window.
  - If the forecast window lies within the downloaded data range, the actual (true) values are also plotted.
- To fetch new power generation data, the script uses the function `power_api_call()` defined inside `real_time_power_api.py`.
- Weather data is retrieved via the function `weather_api_call()` defined inside `real_time_weather_api.py`. 
- Downloaded data (if API access were available) is not saved permanently.

### ⚠️ Warnings

- We currently **lack power data between 2024-01-01 and 2024-06-20**.
- The **API key for real-time power generation data is unavailable**, so forecast windows must be **within the downloaded dataset** ending on `2025-06-22`.


In [1]:
from kNN_real_time_script import kNN_forecast

#### Two examples of interactive user input: call the method `input()`


In [5]:
knn = kNN_forecast()
knn.input()
knn.forecast()


Please enter forecast start date in the format YYYY-MM-DD.The date should be in between 2024-08-20 to 2025-06-26.

You have entered: 2025-03-01.

Please enter forecast window (in days). It should be an integer between 1 or 30.

You have entered: 15

Your forecast window is days between: 2025-03-01 and 2025-03-15

Duplicates or NaN found in power data. Processing and cleaning data...
Please wait. Plotting your forecast...


With manual input training window length:

In [7]:
knn = kNN_forecast(days_to_train_on=100)
knn.input()
knn.forecast()


Please enter forecast start date in the format YYYY-MM-DD.The date should be in between 2024-09-29 to 2025-06-26.

You have entered: 2024-12-31.

Please enter forecast window (in days). It should be an integer between 1 or 30.

You have entered: 2

Your forecast window is days between: 2024-12-31 and 2025-01-01

Duplicates or NaN found in power data. Processing and cleaning data...
Please wait. Plotting your forecast...


#### One example of manual user input: call the method `manual_input()`


In [6]:
knn = kNN_forecast()
knn.manual_input('2025-06-17', 3)
knn.forecast()


Your forecast window is days between: 2025-06-17 and 2025-06-19

Duplicates or NaN found in power data. Processing and cleaning data...
Please wait. Plotting your forecast...


## To download weather data in `already_downloaded_data`
Used to download weather data between 2024-01-01 till 2025-06-22

In [ ]:
import time
import requests
import pandas as pd
from typing import List, Dict, Optional
import os

#Function to slow down requests to avoid rate limit on the API
def fetch_with_retry(url, params, max_retries=3):
    for _ in range(max_retries):
        response = requests.get(url, params=params)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 10))
            print(f"Rate limited. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            print(f"Error {response.status_code}: {response.text}")
            break
    return None


def fetch_historical_weather_multiple(
    latitudes: List[float],
    longitudes: List[float],
    start_datetime: str,
    end_datetime: str,
    location_names: Optional[List[str]] = None,
) -> Dict[str, pd.DataFrame]:
    """
    Fetch historical weather data for multiple lat/lon pairs, including wind direction.

    Args:
        latitudes: List of latitudes.
        longitudes: List of longitudes.
        start_date: Start date in "YYYY-MM-DD" format.
        end_date: End date in "YYYY-MM-DD" format.
        location_names: Optional names for each location (default: "loc_0", "loc_1", ...).

    Returns:
        Dictionary of DataFrames (key: location name, value: weather data).
    """
    start_date = start_datetime.split("T")[0]
    end_date = end_datetime.split("T")[0]

    if len(latitudes) != len(longitudes):
        raise ValueError("Latitudes and longitudes must have the same length.")
    
    if location_names is None:
        location_names = [f"loc_{i}" for i in range(len(latitudes))]
    elif len(location_names) != len(latitudes):
        raise ValueError("Location names must match latitudes/longitudes length.")

    weather_data = {}
    base_url = "https://archive-api.open-meteo.com/v1/archive"         ## for archived data
    #base_url = "https://api.open-meteo.com/v1/forecast"                 ## for forcasted data

    for lat, lon, name in zip(latitudes, longitudes, location_names):
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": [
                "temperature_2m",
                "wind_speed_10m",
                "relative_humidity_2m"
            ],
            "timezone": "UTC"
        }
        
        data = fetch_with_retry(base_url, params=params,max_retries=5)

        df = pd.DataFrame(data["hourly"])
        df['time'] = pd.to_datetime(df['time'], utc=True)
        time_filter = (df["time"] >= start_datetime) & (df["time"] <= end_datetime)
        df["location"] = name 
        df = df[time_filter]
        df['time'] = df['time'] - pd.Timedelta(hours=5)
        weather_data[name] = df

    return weather_data

## Setting up the required varibales to call the above function
position_df = pd.read_csv(r'../../data/raw_data/hydroquebec_wind_farms_in_service.csv')
# Example: Montreal (45.5017° N, 73.5673° W)
latitude = position_df['latitude']
longitude = position_df['longitude']
start_date = '2024-01-01'
end_date = '2025-06-23'
start_date += "T05:00:00"     ## adjusting for the time-zone to UTC conversion     
end_date += "T04:00:00"
weather_data = fetch_historical_weather_multiple(latitude, longitude, start_date, 
                                                     end_date,location_names=position_df['name'])

In [ ]:

keys = [x for x in weather_data.keys()]
merge_wdf = pd.DataFrame()
for key, label in zip(keys, position_df["labels"]):
    weather_df = pd.DataFrame(weather_data[key])
    
    new_columns = [f"{column}" for column in weather_df.columns[0:1]] + [f"{column}_{label}" for column in weather_df.columns[1:]]
    weather_df.columns = new_columns
    weather_df = weather_df.drop(columns=[new_columns[-1]])                    ## drop the location, no need anymore
    weather_df['time'] = pd.to_datetime(weather_df['time'])                 ## convert time to pandas timestamp
    weather_df = weather_df.sort_values('time')   
    if key == keys[0]:
        merge_wdf = weather_df
    else:
        merge_wdf = pd.merge(merge_wdf, weather_df, on=['time'])
merge_wdf = merge_wdf.dropna()
merge_wdf.to_csv("already_downloaded_data/weather_data.csv")